# Numba Basics

Numba is a Just-In-Time (JIT) compiler of Python functions.  It translates a Python function when it is called into a machine code equivalent that runs anywhere from 2x (simple NumPy operations) to 100x (complex Python loops) faster.  In this notebook, we show some basic examples of using Numba.

In [1]:
import numpy as np
import numba
from numba import jit
import time

### Let's see a computation using Numpy

In [2]:
N = 10_000_000
arr = np.random.rand(N)

def python_loop(arr):
    result = np.zeros_like(arr)
    for i in range(len(arr)):
        result[i] = arr[i] ** 2
    return result

start = time.time()
python_loop(arr)
print("Python loop:", time.time() - start)

Python loop: 4.500308036804199


### Using numba

In [3]:
from numba import njit

@njit
def numba_loop(arr):
    result = np.zeros_like(arr)
    for i in range(len(arr)):
        result[i] = arr[i] ** 2
    return result

# First run (compilation)
numba_loop(arr)

start = time.time()
numba_loop(arr)
print("Numba loop:", time.time() - start)

Numba loop: 0.060135602951049805


Numba uses Python *decorators* to transform Python functions into functions that compile themselves.  The most common Numba decorator is `@jit`, which creates a normal function for execution on the CPU.

Numba works best on numerical functions that make use of NumPy arrays.  Here's an example:

In [5]:
@jit(nopython=True)
def go_fast(a): # Function is compiled to machine code when called the first time
    trace = 0.0
    # assuming square input matrix
    for i in range(a.shape[0]):   # Numba likes loops
        trace += np.tanh(a[i, i]) # Numba likes NumPy functions
    return a + trace              # Numba likes NumPy broadcasting

The `nopython=True` option requires that the function be fully compiled (so that the Python interpreter calls are completely removed), otherwise an exception is raised.  These exceptions usually indicate places in the function that need to be modified in order to achieve better-than-Python performance.  We strongly recommend always using `nopython=True`.

The function has not yet been compiled.  To do that, we need to call the function:

In [6]:
x = np.arange(100).reshape(10, 10)
go_fast(x)

array([[  9.,  10.,  11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.],
       [ 19.,  20.,  21.,  22.,  23.,  24.,  25.,  26.,  27.,  28.],
       [ 29.,  30.,  31.,  32.,  33.,  34.,  35.,  36.,  37.,  38.],
       [ 39.,  40.,  41.,  42.,  43.,  44.,  45.,  46.,  47.,  48.],
       [ 49.,  50.,  51.,  52.,  53.,  54.,  55.,  56.,  57.,  58.],
       [ 59.,  60.,  61.,  62.,  63.,  64.,  65.,  66.,  67.,  68.],
       [ 69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,  77.,  78.],
       [ 79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,  88.],
       [ 89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.],
       [ 99., 100., 101., 102., 103., 104., 105., 106., 107., 108.]])

This first time the function was called, a new version of the function was compiled and executed.  If we call it again, the previously generated function executions without another compilation step.

In [8]:
go_fast(2*x)

array([[  9.,  11.,  13.,  15.,  17.,  19.,  21.,  23.,  25.,  27.],
       [ 29.,  31.,  33.,  35.,  37.,  39.,  41.,  43.,  45.,  47.],
       [ 49.,  51.,  53.,  55.,  57.,  59.,  61.,  63.,  65.,  67.],
       [ 69.,  71.,  73.,  75.,  77.,  79.,  81.,  83.,  85.,  87.],
       [ 89.,  91.,  93.,  95.,  97.,  99., 101., 103., 105., 107.],
       [109., 111., 113., 115., 117., 119., 121., 123., 125., 127.],
       [129., 131., 133., 135., 137., 139., 141., 143., 145., 147.],
       [149., 151., 153., 155., 157., 159., 161., 163., 165., 167.],
       [169., 171., 173., 175., 177., 179., 181., 183., 185., 187.],
       [189., 191., 193., 195., 197., 199., 201., 203., 205., 207.]])

To benchmark Numba-compiled functions, it is important to time them without including the compilation step, since the compilation of a given function will only happen once for each set of input types, but the function will be called many times.

In a notebook, the `%timeit` magic function is the best to use because it runs the function many times in a loop to get a more accurate estimate of the execution time of short functions.

In [9]:
%timeit go_fast(x)

1.46 µs ± 462 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


Let's compare to the uncompiled function.  Numba-compiled function have a special `.py_func` attribute which is the original uncompiled Python function.  We should first verify we get the same results:

This function checks that two arrays are exactly the same:

1. Same shape
2. Same elements
3. Same order

If they are equal → nothing happens
If they differ → it raises an AssertionError

In [10]:
np.testing.assert_array_equal(go_fast(x), go_fast.py_func(x))

And test the speed of the Python version:

In [11]:
%timeit go_fast.py_func(x)

22.4 µs ± 826 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


The original Python function is more than 20x slower than the Numba-compiled version.  However, the Numba function used explicit loops, which are very fast in Numba and not very fast in Python.  Our example function is so simple, we can create an alternate version of `go_fast` using only NumPy array expressions:

In [12]:
def go_numpy(a):
    return a + np.tanh(np.diagonal(a)).sum()

In [13]:
np.testing.assert_array_equal(go_numpy(x), go_fast(x))

In [14]:
%timeit go_numpy(x)

7.47 µs ± 76.5 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


The NumPy version is more than 2x faster than Python, but still 10x slower than Numba.

### Supported Python Features

Numba works best when used with NumPy arrays, but Numba also supports other data types out of the box:

* `int`, `float`
* `tuple`, `namedtuple`
* `list` (with some restrictions)
* ... and others.  See the [Reference Manual](https://numba.pydata.org/numba-doc/latest/reference/pysupported.html) for more details.

In particular, tuples are useful for returning multiple values from functions:

When Numba is translating Python to machine code, it uses the [LLVM](https://llvm.org/) library to do most of the optimization and final code generation.  This automatically enables a wide range of optimizations that you don't even have to think about.

In [15]:
N = 10_000_000
x = np.random.random(N)
y = np.random.random(N)

# NumPy
def numpy_version(x, y):
    return 2 * x + y**2

# Numba
@njit
def numba_version(x, y):
    # Numba looks at this expression and generates a single machine-code loop
    return 2 * x + y**2

# --- Benchmarking ---
# Warm up Numba (compilation happens on first call)
numba_version(x, y)

print('Numpy: ', end="")
timer_np = %timeit numpy_version(x, y)
print('Numba: ', end="")
timer_nb = %timeit numba_version.py_func(x, y)


Numpy: 69 ms ± 2.93 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
Numba: 70.9 ms ± 4.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In the example below, we calculate a simple polynomial-like expression: $z = 2x + y^2$.

NumPy Version: It first calculates $y^2$ (creates a temporary array), then $2x$ (creates another temporary), then adds them (creates a final array). That’s three passes over memory.

Numba Version: It "fuses" these into a single loop. It takes one element of $x$ and $y$, does all the math, and sticks the result in the output. One pass.

# Comparison with Cython

In [16]:
# We will simulate a random walk of 10 million steps
n_steps = 10_000_000
data = np.random.randn(n_steps)

### Cython Approach

In [17]:
# First, load the Cython extension
%load_ext Cython

In [18]:

%%cython
import numpy as np
cimport numpy as l_np
cimport cython

@cython.boundscheck(False) # Pro-tip: Disabling checks makes it even faster!
@cython.wraparound(False)
def cython_random_walk(double[:] steps):
    cdef int n = steps.shape[0]
    cdef double[:] path = np.zeros(n, dtype=np.float64)
    cdef double current_pos = 0.0
    cdef int i

    for i in range(n):
        current_pos += steps[i]
        path[i] = current_pos
    return path

Content of stderr:
In file included from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /root/.cache/ipython/cython/_cython_magic_d496aa594971183164fae51dd6cd8c959f404497.c:1250:
/usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~

### Numba approach

In [19]:
@njit
def numba_random_walk(steps):
    n = steps.shape[0]
    path = np.zeros(n, dtype=np.float64)
    current_pos = 0.0

    for i in range(n):
        current_pos += steps[i]
        path[i] = current_pos
    return path

## Comparison

In [20]:
print("Timing Cython...")
%timeit cython_random_walk(data)

print("\nTiming Numba...")
# Note: The first run of Numba includes compile time,
# so we run it once before timing or use %timeit which runs multiple times.
%timeit numba_random_walk(data)

# For context, show them how slow regular Python is:
def python_random_walk(steps):
    path = np.zeros(len(steps))
    current_pos = 0.0
    for i in range(len(steps)):
        current_pos += steps[i]
        path[i] = current_pos
    return path

print("\nTiming Standard Python...")
%timeit python_random_walk(data)

Timing Cython...
62.4 ms ± 6.89 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Timing Numba...
64.8 ms ± 1.84 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Timing Standard Python...
2.81 s ± 427 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# The Custom KNN Challenge

Write a function to find the "Closest Point" in a dataset of 100,000 items.

Level 1: Write it in pure Python (It will be very slow).

Level 2: Use @njit on that same function (It will be 100x faster).

Level 3: Use @njit(parallel=True) and prange to use all CPU cores (Even faster!).

In [21]:
from numba import prange

# 100,000 points in 2D space (x, y)
ref_points = np.random.random((100000, 2))
query_point = np.array([0.5, 0.5])

In [22]:
def python_knn(query, refs):
    distances = np.zeros(len(refs))
    for i in range(len(refs)):
        # Euclidean Distance: sqrt((x2-x1)^2 + (y2-y1)^2)
        dist = ((refs[i][0] - query[0])**2 + (refs[i][1] - query[1])**2)**0.5
        distances[i] = dist
    return distances

In [23]:
@njit
def numba_knn(query, refs):
    distances = np.zeros(len(refs))
    for i in range(len(refs)):
        dist = ((refs[i][0] - query[0])**2 + (refs[i][1] - query[1])**2)**0.5
        distances[i] = dist
    return distances

In [24]:
@njit(parallel=True)
def numba_parallel_knn(query, refs):
    distances = np.zeros(len(refs))
    for i in prange(len(refs)): # prange runs this in parallel!
        dist = ((refs[i][0] - query[0])**2 + (refs[i][1] - query[1])**2)**0.5
        distances[i] = dist
    return distances

`@njit(Parallel=True)` decorator in Numba is used to automatically parallelize Python code, allowing numerical loops and array operations to run across multiple CPU cores. It enables ``no-python`` mode (strict compilation) and tells the Numba compiler to identify opportunities for parallel execution, such as in array expressions, reductions, and loops designed with prange

In [25]:
print("--- Starting the KNN Speed Challenge ---")

start = time.time()
python_knn(query_point, ref_points)
print(f"Pure Python: {time.time() - start:.4f} seconds")

# Run once to compile, then time it
numba_knn(query_point, ref_points)
start = time.time()
numba_knn(query_point, ref_points)
print(f"Numba (Standard): {time.time() - start:.4f} seconds")

# Run once to compile, then time it
numba_parallel_knn(query_point, ref_points)
start = time.time()
numba_parallel_knn(query_point, ref_points)
print(f"Numba (Parallel): {time.time() - start:.4f} seconds")

--- Starting the KNN Speed Challenge ---
Pure Python: 0.1487 seconds
Numba (Standard): 0.0005 seconds
Numba (Parallel): 0.0006 seconds


# Gradient Descent

In [26]:
# Create 1 million data points
n_points = 1_000_000
x = np.random.random(n_points)
y = 2.5 * x + 1.0 + np.random.normal(0, 0.1, n_points) # True m=2.5, c=1.0

# Initialize random weights
m, c = 0.0, 0.0
learning_rate = 0.1
epochs = 100

In [27]:
def python_gradient_descent(x, y, m, c, lr, epochs):
    n = len(x)
    for _ in range(epochs):
        dm = 0
        dc = 0
        for i in range(n):
            # Calculate error for one point
            prediction = m * x[i] + c
            error = prediction - y[i]
            # Accumulate gradients
            dm += error * x[i]
            dc += error

        # Update weights
        m -= (2/n) * dm * lr
        c -= (2/n) * dc * lr
    return m, c

In [28]:
@njit
def numba_gradient_descent(x, y, m, c, lr, epochs):
    n = len(x)
    for _ in range(epochs):
        dm = 0
        dc = 0
        for i in range(n):
            prediction = m * x[i] + c
            error = prediction - y[i]
            dm += error * x[i]
            dc += error

        m -= (2/n) * dm * lr
        c -= (2/n) * dc * lr
    return m, c

In [29]:
print("Starting Gradient Descent Race...")

# Numba "Warm-up" (Compiling)
numba_gradient_descent(x, y, m, c, learning_rate, 1)

start = time.time()
m_res, c_res = numba_gradient_descent(x, y, m, c, learning_rate, epochs)
print(f"Numba Time: {time.time() - start:.4f} seconds")
print(f"Result: m={m_res:.2f}, c={c_res:.2f}")

# Warning: Python will be VERY slow here.
# For a live demo, maybe reduce 'epochs' to 1 for the Python version.
start = time.time()
m_py, c_py = python_gradient_descent(x, y, m, c, learning_rate, 5) # Only 5 epochs
print(f"Python Time (for only 5 epochs!): {time.time() - start:.4f} seconds")

Starting Gradient Descent Race...
Numba Time: 0.1741 seconds
Result: m=2.09, c=1.22
Python Time (for only 5 epochs!): 5.1324 seconds
